In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import akshare as ak
import baostock as bs

\

d:\software\Anaconda\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\software\Anaconda\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_excel("399317_cons.xls", engine="xlrd")
df.head()

,日期,样本代码,样本简称,所属行业,权重（%）
0,2026-02-06,300750,宁德时代,工业,1.77
1,2026-02-06,600519,贵州茅台,主要消费,1.6
2,2026-02-06,601318,中国平安,金融,1.26
3,2026-02-06,601899,紫金矿业,原材料,1.04
4,2026-02-06,600036,招商银行,金融,0.9


In [5]:
def normalize_stock_code(code):
    """
    A 股股票代码统一成 akshare / wind 可用格式
    """
    code = str(code).strip().zfill(6)

    if code[0] in ["6", "9"]:
        return "sh." + code
    elif code[0] in ["0", "3"]:
        return "sz." + code
    else:
        raise ValueError(f"未知股票代码：{code}")


In [6]:
stock_list = df["样本代码"].apply(normalize_stock_code).tolist()
stock_list

['sz.300750',
 'sh.600519',
 'sh.601318',
 'sh.601899',
 'sh.600036',
 'sz.300308',
 'sz.000333',
 'sz.300502',
 'sh.600900',
 'sh.601166',
 'sz.300059',
 'sh.600030',
 'sz.002475',
 'sh.601398',
 'sh.600276',
 'sz.002594',
 'sh.601211',
 'sh.688041',
 'sh.688981',
 'sh.688256',
 'sz.300274',
 'sh.603259',
 'sh.601288',
 'sz.000858',
 'sh.601138',
 'sz.002371',
 'sh.688008',
 'sh.603986',
 'sz.000651',
 'sh.600309',
 'sh.600887',
 'sh.601328',
 'sz.300476',
 'sh.688012',
 'sh.603993',
 'sz.300124',
 'sz.000725',
 'sh.600919',
 'sh.601601',
 'sh.600000',
 'sh.600150',
 'sh.600031',
 'sh.601857',
 'sh.601816',
 'sz.000338',
 'sz.002415',
 'sh.600089',
 'sh.601688',
 'sz.002714',
 'sh.600111',
 'sh.601088',
 'sz.300760',
 'sz.000063',
 'sz.002602',
 'sh.601012',
 'sz.300394',
 'sz.002142',
 'sh.603019',
 'sz.002050',
 'sh.603799',
 'sz.002028',
 'sz.002230',
 'sz.000792',
 'sh.601600',
 'sz.000425',
 'sh.600016',
 'sh.600690',
 'sh.603501',
 'sz.002463',
 'sz.000001',
 'sz.002352',
 'sh.6

In [7]:
import baostock as bs
import pandas as pd
from tqdm import tqdm
import time

# ========= 1. 登录 =========
lg = bs.login()
print("login error_code:", lg.error_code)
print("login error_msg:", lg.error_msg)

# ========= 2. 读取股票列表 =========
# stock_list = ["sh.600519", "sz.300750", ...]


# ========= 3. 参数设置 =========
START_DATE = "2010-01-01"
END_DATE   = "2025-12-31"

FIELDS = (
    "date,code,open,high,low,close,volume,amount,"
    "adjustflag,turn,tradestatus,pctChg,isST"
)

all_data = []

# ========= 4. 循环拉取 =========
for stock in tqdm(stock_list):
    try:
        rs = bs.query_history_k_data_plus(
            stock,
            FIELDS,
            start_date=START_DATE,
            end_date=END_DATE,
            frequency="d",
            adjustflag="3"   # 3=后复权（推荐）
        )

        if rs.error_code != "0":
            print(f"{stock} 查询失败：{rs.error_msg}")
            continue

        data_list = []
        while rs.next():
            data_list.append(rs.get_row_data())

        if not data_list:
            continue

        df = pd.DataFrame(data_list, columns=rs.fields)

        # 类型处理（非常重要）
        df["date"] = pd.to_datetime(df["date"])
        numeric_cols = [
            "open","high","low","close","volume",
            "amount","turn","pctChg"
        ]
        df[numeric_cols] = df[numeric_cols].apply(
            pd.to_numeric, errors="coerce"
        )

        all_data.append(df)

        # 防止请求过快
        time.sleep(0.05)

    except Exception as e:
        print(f"{stock} 异常：{e}")

# ========= 5. 合并 =========
price_df = pd.concat(all_data, ignore_index=True)
price_df = price_df.sort_values(["code", "date"]).reset_index(drop=True)

# ========= 6. 保存 =========
price_df.to_csv(
    "A_stock_daily_2010_2025_hfq.csv",
    index=False
)

print("完成，总行数：", len(price_df))

# ========= 7. 登出 =========
bs.logout()


login success!
login error_code: 0
login error_msg: success


100%|██████████| 5252/5252 [35:28<00:00,  2.47it/s]  


完成，总行数： 12320871
logout success!


In [12]:
df = pd.read_csv("A_stock_daily_2010_2025_hfq.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["code", "date"])

# 只保留正常交易、非 ST
df = df[(df["tradestatus"] == 1) & (df["isST"] == 0)]
df.head()

,date,code,open,high,low,close,volume,amount,adjustflag,turn,tradestatus,pctChg,isST
0,2010-01-04,sh.600000,21.83,21.87,21.16,21.19,66191338.0,1.419984e+09,3,0.835129,1,-2.3052,0
1,2010-01-05,sh.600000,21.41,21.58,20.79,21.35,115147943.0,2.436891e+09,3,1.452808,1,0.7551,0
2,2010-01-06,sh.600000,21.29,21.30,20.88,20.93,96782575.0,2.034174e+09,3,1.221095,1,-1.9672,0
3,2010-01-07,sh.600000,20.88,21.04,20.30,20.46,85236072.0,1.761801e+09,3,1.075414,1,-2.2456,0
4,2010-01-08,sh.600000,20.34,20.80,20.30,20.69,65707646.0,1.349532e+09,3,0.829026,1,1.1241,0


In [38]:
# 1）个股层面：是否创20日新高
df["rolling_max_20"] = (
    df.groupby("code")["close"]
      .rolling(20, min_periods=20)
      .max()
      .reset_index(level=0, drop=True)
)

df["is_new_high_20"] = (df["close"] == df["rolling_max_20"]).astype(int)

# 2）市场层面：当日创新高股票占比
daily_new_high_ratio = (
    df.groupby("date")["is_new_high_20"]
      .mean()
      .rename("new_high_ratio")
)
daily_new_high_ratio

date
2010-01-04    0.000000
2010-01-05    0.000000
2010-01-06    0.000000
2010-01-07    0.000000
2010-01-08    0.000000
                ...   
2025-12-25    0.175986
2025-12-26    0.122638
2025-12-29    0.124422
2025-12-30    0.115400
2025-12-31    0.110932
Name: new_high_ratio, Length: 3886, dtype: float64

In [60]:
# ===== 1) 读入数据（国证A指 399317）=====
index_df = pd.read_csv("index_hist_cni_399317.csv", parse_dates=["日期"])
index_df = index_df.sort_values("日期").set_index("日期")

# ===== 2) 中文列名 -> 英文列名（后续代码就不用大改）=====
index_df = index_df.rename(columns={
    "开盘价": "open",
    "最高价": "high",
    "最低价": "low",
    "收盘价": "close",
    "涨跌幅": "pct_chg",
    "成交量": "volume",
    "成交额": "amount",
})

# ===== 3) 用 close 计算日收益 ret（你原来这行保留即可）=====

close = pd.Series(index_df['close'], index=index_df.index)
print(close[0:5])
index_ret = close.pct_change().fillna(0)

index_df.head()

日期
2010-01-04    3276.2273
2010-01-05    3305.7476
2010-01-06    3289.3477
2010-01-07    3224.5501
2010-01-08    3243.8944
Name: close, dtype: float64


,open,high,low,close,pct_chg,volume,amount
日期,,,,,,,
2010-01-04,3311.7279,3316.1241,3276.2273,3276.2273,-0.0060,15678.34,2069.70
2010-01-05,3283.1592,3309.1449,3245.8613,3305.7476,0.0090,18735.01,2549.08
2010-01-06,3301.4769,3327.4824,3288.7061,3289.3477,-0.0050,18338.14,2503.23
2010-01-07,3287.4944,3299.9601,3208.0001,3224.5501,-0.0197,18508.14,2480.13
2010-01-08,3210.6486,3244.4500,3189.2038,3243.8944,0.0060,14506.98,1924.86


In [61]:
# 将 daily_new_high_ratio 对齐到指数日期（使用 index_df 的索引）并填充缺失值
if 'index_df' in globals():
    factor = daily_new_high_ratio.reindex(index_df.index).fillna(0)
else:
    factor = daily_new_high_ratio.fillna(0)



In [62]:
UP = 0.5
DOWN = 0.2

# =========================
# 参数（你可以按需改）
# =========================
START = "2010-01-01"
END   = "2025-12-31"

FACTOR_N = 20          # 20日价格乖离率
SMOOTH_N = 60          # 因子平滑窗口（为了更稳健，且持仓天数更接近研报示例）
LOOKBACK = 5           # 判断“升高/下降”的回看天数
FEE_RATE = 0.0005      # 单边费率 0.05% = 0.0005（双边都收）

PLOT_START = "2020-01-01"   # 图11/图13的展示区间
PLOT_END   = END

signal = pd.Series(np.nan, index=factor.index)

signal[factor >= UP] = 1      # 市场强势
signal[factor <= DOWN] = 0   # 市场转弱

signal = signal.ffill().fillna(0).astype(int)


In [63]:
pos = signal.shift(1).fillna(0).astype(int)
pos_hold = pos.shift(1).fillna(0)

gross_ret = pos_hold * index_ret

turnover = (pos - pos.shift(1)).abs().fillna(0)
cost = turnover * FEE_RATE
strat_ret = (1 + gross_ret) * (1 - cost) - 1

nav = (1 + strat_ret).cumprod()
dd = nav / nav.cummax() - 1

bt = pd.DataFrame({
    "close": close,
    "factor": factor,
    "signal": signal,
    "pos": pos,
    "pos_hold": pos_hold,
    "turnover": turnover,
    "cost": cost,
    "index_ret": index_ret,
    "strat_ret": strat_ret,
    "nav": nav,
    "dd": dd,
})


In [64]:
TRADING_DAYS = 252
r = bt["strat_ret"].dropna()
# 使用 nav 相对起始值计算总收益与年化收益（CAGR）
n_days = len(nav)
start_nav = nav.iloc[0]
end_nav = nav.iloc[-1]
total_return = end_nav / start_nav - 1
ann_ret = (end_nav / start_nav) ** (TRADING_DAYS / n_days) - 1

# 波动与夏普/Sortino
ann_vol = r.std(ddof=0) * np.sqrt(TRADING_DAYS)
sharpe = r.mean() / r.std(ddof=0) * np.sqrt(TRADING_DAYS) if r.std(ddof=0) != 0 else np.nan
try:
    import empyrical as ep
    sortino = ep.sortino_ratio(r)
    downside_risk = ep.downside_risk(r)
    emp_max_dd = ep.max_drawdown(r)
except Exception:
    sortino = np.nan
    downside_risk = np.nan
    emp_max_dd = np.nan

# 最大回撤（使用净值序列）
max_dd = dd.min()
calmar = ann_ret / abs(max_dd) if max_dd != 0 else np.nan

# 持仓相关指标（基于 bt['pos']）
pos = bt.get('pos') if 'bt' in globals() else None
if pos is None:
    mean_hold = np.nan
    median_hold = np.nan
    n_trades = np.nan
else:
    pos = pos.fillna(0).astype(int)
    # 持仓连续区间 id
    run_id = (pos != pos.shift(1)).cumsum()
    run_sizes = pos.groupby(run_id).sum()
    holding_runs = run_sizes[run_sizes > 0]
    mean_hold = float(holding_runs.mean()) if len(holding_runs) > 0 else 0.0
    median_hold = float(holding_runs.median()) if len(holding_runs) > 0 else 0.0
    n_trades = int(len(holding_runs))

# 胜率与盈亏比（基于策略日收益）
strat = bt['strat_ret']
if pos is not None:
    wins = strat[pos == 1]
else:
    wins = strat
win_rate_in_pos = float((wins > 0).mean()) if len(wins) > 0 else np.nan
avg_win = float(wins[wins > 0].mean()) if (wins > 0).any() else np.nan
avg_loss = float(wins[wins < 0].mean()) if (wins < 0).any() else np.nan
payoff = (avg_win / -avg_loss) if (not np.isnan(avg_win) and not np.isnan(avg_loss) and avg_loss != 0) else np.nan

# 交易成本与换手（若存在）
turnover = bt['turnover'] if 'turnover' in bt.columns else None
total_turnover = float(turnover.sum()) if turnover is not None else np.nan
total_cost = float(bt['cost'].sum()) if 'cost' in bt.columns else np.nan

# 汇总表（DataFrame）并导出
summary = {
    '总收益': total_return,
    '年化收益(CAGR)': ann_ret,
    '年化波动': ann_vol,
    '夏普比率': sharpe,
    'Sortino比率': sortino,
    '最大回撤': max_dd,
    'empyrical_max_dd': emp_max_dd,
    'Calmar比率': calmar,
    '平均持仓天数': mean_hold,
    '持仓天数中位数': median_hold,
    '持仓次数': n_trades,
    '持仓胜率（在仓）': win_rate_in_pos,
    '平均盈利(单日)': avg_win,
    '平均亏损(单日)': avg_loss,
    '盈亏比(平均盈利/平均亏损)': payoff,
    '总换手': total_turnover,
    '总交易费用': total_cost,
}
summary_df = pd.DataFrame(summary, index=['strategy'])
summary_df.to_csv('backtest_summary.csv', encoding='utf-8-sig')

# 每日明细表：合并 bt（假定包含 strat_ret, index_ret, nav, dd, signal 等列）
daily = bt.copy()
# 如果存在 date 列，设置为索引，保证为 DatetimeIndex
if 'date' in daily.columns:
    daily['date'] = pd.to_datetime(daily['date'])
    daily = daily.set_index('date')

# 若有 daily_new_high_ratio，将其合并
if 'daily_new_high_ratio' in globals():
    if isinstance(daily_new_high_ratio, pd.DataFrame):
        daily_new = daily_new_high_ratio.copy()
        daily_new.index = pd.to_datetime(daily_new.index)
        daily = daily.join(daily_new[['new_high_ratio','close']], how='left')
    else:
        daily = daily.join(daily_new_high_ratio.rename('new_high_ratio'), how='left')

# 导出每日明细
daily.to_csv('backtest_daily.csv', encoding='utf-8-sig')
try:
    daily.to_excel('backtest_daily.xlsx')
except Exception:
    pass
summary_df
# print('Saved backtest_summary.csv and backtest_daily.csv')

<ipython-input-64-04c10fb20d05>:101: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.1' currently installed).
  daily.to_excel('backtest_daily.xlsx')


,总收益,年化收益(CAGR),年化波动,夏普比率,Sortino比率,最大回撤,empyrical_max_dd,Calmar比率,平均持仓天数,持仓天数中位数,持仓次数,持仓胜率（在仓）,平均盈利(单日),平均亏损(单日),盈亏比(平均盈利/平均亏损),总换手,总交易费用
strategy,0.261412,0.015174,0.058538,0.286473,0.424416,-0.099038,-0.099038,0.153212,4.230769,4.0,26,0.4,0.014989,-0.008002,1.873084,52.0,0.026


In [69]:
import matplotlib
from matplotlib import font_manager
# 设置中文字体
font_path = "C:/Windows/Fonts/simhei.ttf"  # 替换为系统中实际存在的中文字体路径
font_prop = font_manager.FontProperties(fname=font_path)
matplotlib.rcParams['font.family'] = font_prop.get_name()
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

In [72]:
# 绘图：使用 .values 替代 .index 避免 MultiIndex 问题
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(close.index, label="国证A指", linewidth=1.2)
ax1.set_ylabel("Index")

ax2 = ax1.twinx()
ax2.plot(factor.values, color="gray", alpha=0.6, label="20日创新高占比")
ax2.set_ylabel("20日创新高占比")

ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title("国证A指 vs 20日创新高股票占比")
plt.tight_layout()
plt.show()

ValueError: Input could not be cast to an at-least-1D NumPy array

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(close.va, close, linewidth=1.2)
ax1.set_ylabel("Index")

ax2 = ax1.twinx()
ax2.bar(signal.index, signal, alpha=0.25)
ax2.set_ylim(-0.1, 1.1)
ax2.set_ylabel("Signal")

plt.title("20日创新高占比择时信号")
plt.tight_layout()
plt.show()
